# DisputeCourt — GRPO training (Colab)

Run this on a T4/A100 runtime: **Runtime > Change runtime type > GPU**.

This trains a small model (Qwen2.5-0.5B-Instruct by default) to output a calibrated verdict for Visa 13.1 chargeback cases, using the same reward function validated in `training/test_reward_sanity.py` on your local repo. If you haven't run that sanity check yet, do it locally first — it takes seconds and catches a bad reward shape before you spend GPU time on it.

In [ ]:
# Colab ships torchao 0.10.0; recent peft requires >=0.16.0.
# Upgrade torchao first, then pin peft. AFTER this cell: Runtime > Restart session,
# then run from the clone cell onward (do not re-run this pip cell after restart).
!pip install -q --upgrade "torchao>=0.16.0"
!pip install -q trl peft transformers bitsandbytes accelerate datasets

import torchao, peft
print("torchao", torchao.__version__, "peft", peft.__version__)
assert tuple(int(x) for x in torchao.__version__.split(".")[:2]) >= (0, 16), (
    "torchao still < 0.16 — Runtime > Restart session, then re-run this cell once."
)

## Get your data in

Two options — use whichever is easier:

**Option A: clone your GitHub repo** (recommended once you've pushed — this is also good practice for the actual submission). Set `GITHUB_REPO_URL` below.

**Option B: upload the data file directly.** Leave `GITHUB_REPO_URL` empty and run the upload cell instead.

In [ ]:
GITHUB_REPO_URL = "https://github.com/AniketAslaliya/disputecourt.git"

import os

if GITHUB_REPO_URL:
    !git clone {GITHUB_REPO_URL} repo
    DATA_PATH = "repo/data/train.jsonl"
    EVAL_PATH = "repo/data/eval.jsonl"
    if not os.path.exists(DATA_PATH):
        print(f"WARNING: {DATA_PATH} not found -- falling back to generated_cases.jsonl")
        DATA_PATH = "repo/data/generated_cases.jsonl"
        EVAL_PATH = None
else:
    DATA_PATH = None
    EVAL_PATH = None

In [ ]:
# Only run this cell if GITHUB_REPO_URL above was left empty.
if not GITHUB_REPO_URL:
    from google.colab import files
    uploaded = files.upload()  # pick generated_cases.jsonl or seed_cases_labeled.jsonl from your machine
    DATA_PATH = list(uploaded.keys())[0]

print(f"Using data file: {DATA_PATH}")

## Reward function

Copied inline from `training/reward.py` so this notebook is self-contained. If you change the reward design locally, re-copy it here — don't let these two drift silently out of sync.

In [ ]:
import json
import re

CORRECTNESS_WEIGHT = 1.0
CALIBRATION_WEIGHT = 0.5
ABSTENTION_WEIGHT = 0.3
VALID_VERDICTS = {"represent", "accept", "abstain"}


def parse_completion(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    if parsed.get("verdict") not in VALID_VERDICTS:
        return None
    conf = parsed.get("confidence")
    if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
        return None
    return {"verdict": parsed["verdict"], "confidence": float(conf)}


def compute_reward(completion_text, true_verdict):
    parsed = parse_completion(completion_text)
    if parsed is None:
        return -1.5
    pred_verdict = parsed["verdict"]
    pred_conf = parsed["confidence"]
    is_correct = pred_verdict == true_verdict
    correctness_term = CORRECTNESS_WEIGHT * (1.0 if is_correct else -1.0)
    correctness_indicator = 1.0 if is_correct else 0.0
    calibration_term = -CALIBRATION_WEIGHT * (pred_conf - correctness_indicator) ** 2
    abstention_term = 0.0
    if true_verdict == "abstain" and pred_verdict == "abstain":
        abstention_term = ABSTENTION_WEIGHT
    elif true_verdict != "abstain" and pred_verdict == "abstain":
        abstention_term = -ABSTENTION_WEIGHT * 0.5
    return correctness_term + calibration_term + abstention_term


def trl_reward_fn(completions, true_verdicts, **kwargs):
    return [compute_reward(c, v) for c, v in zip(completions, true_verdicts)]


print("Reward function loaded.")

## Prompt template + dataset

In [ ]:
from datasets import Dataset

SKILL_MD_RULES_COMPACT = """Apply this rules matrix to decide the verdict:
REPRESENT if: (E1 or E2 present) AND (E3 or E5 or E6 present) AND not contradicted
ACCEPT if: (E1 and E2 both absent) OR contradicted
ABSTAIN if: neither resolves cleanly (e.g. delivery present but identity link is weak/partial)

Evidence codes: E1=delivery confirmed, E2=digital delivery/access proof,
E3=AVS address match, E4=signature confirmation, E5=device/card continuity,
E6=employment-at-business-address proof, E7=support communication log."""

PROMPT_TEMPLATE = """You are adjudicating a Visa 13.1 (Merchandise/Services Not Received) chargeback dispute.

{rules}

Case: {narrative}
Evidence on file: {evidence_present}

Output ONLY a JSON object: {{"verdict": "represent"|"accept"|"abstain", "confidence": <float 0-1>, "reasoning": "<short, references specific E-items>"}}"""


def build_prompt(narrative, evidence_present):
    return PROMPT_TEMPLATE.format(rules=SKILL_MD_RULES_COMPACT, narrative=narrative, evidence_present=evidence_present)


rows = [json.loads(line) for line in open(DATA_PATH)]
records = [
    {"prompt": build_prompt(r["narrative"], r["evidence_present"]), "true_verdict": r["verdict"]}
    for r in rows
]
dataset = Dataset.from_list(records)
print(f"Loaded {len(dataset)} examples.")
print(dataset[0]["prompt"][:300])

## Sanity-check the reward on THIS dataset before training

Same check as `training/test_reward_sanity.py`, run here against whatever data you actually loaded above — worth re-confirming since the generated dataset's verdict distribution may differ from the seed set's.

In [ ]:
import random
from collections import Counter

true_verdicts = [r["true_verdict"] for r in records]
dist = Counter(true_verdicts)
total = len(true_verdicts)
print("Ground truth distribution:")
for v, c in dist.most_common():
    print(f"  {v:10s} {c:4d}  ({c/total:.0%})")

def strategy_always_abstain(tv):
    return json.dumps({"verdict": "abstain", "confidence": 0.5})

def strategy_calibrated_80pct(tv):
    if random.random() < 0.8:
        return json.dumps({"verdict": tv, "confidence": random.uniform(0.75, 0.9)})
    other = [v for v in ("represent", "accept", "abstain") if v != tv]
    return json.dumps({"verdict": random.choice(other), "confidence": random.uniform(0.55, 0.7)})

avg_degenerate = sum(compute_reward(strategy_always_abstain(tv), tv) for tv in true_verdicts) / total
avg_good = sum(compute_reward(strategy_calibrated_80pct(tv), tv) for tv in true_verdicts) / total

print(f"\nalways-abstain reward: {avg_degenerate:+.3f}")
print(f"calibrated-80pct reward: {avg_good:+.3f}")
assert avg_good > avg_degenerate, "STOP: degenerate strategy scores higher on this dataset -- fix the reward or the data before training."
print("\nPASS -- safe to train against this dataset.")

## Train

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # swap for a stronger base if you have the GPU budget

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

grpo_config = GRPOConfig(
    output_dir="./checkpoints",
    num_train_epochs=1,
    num_generations=4,
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
)

trainer = GRPOTrainer(
    model=MODEL,
    args=grpo_config,
    train_dataset=dataset,
    peft_config=lora_config,
    reward_funcs=trl_reward_fn,
)

trainer.train()

## Watch for the collapse mode while this runs

If the logged reward flatlines early and stays flat, check a handful of completions manually -- if they're all the same verdict regardless of the case, that's a training-dynamics collapse (different from a reward-shape problem, which the sanity check above already ruled out). This is a legitimate, specific 'what broke' story for the submission form either way -- don't just quietly restart with different hyperparameters without noting what happened.

In [ ]:
trainer.save_model("./checkpoints")
print("Saved. Download ./checkpoints or push it to your GitHub repo (consider git-lfs for the weights).")
print("Next: run this model's outputs through eval/metrics.py and compare against")
print("panel/run_panel_live.py's baseline results using compare_reports().")